# 07 — Reformulação do alvo: alerta antecipado de deterioração

Os notebooks 04 e 06 mostraram que **a vantagem do aprendizado de máquina se concentra na cauda**
— o Random Forest vence no RMSE (sensível a erros grandes) mas perde no MAE (erro típico), e a
diferença no RMSE não é significativa. Havia, portanto, uma incoerência entre o achado e o
instrumento de medida: mediu-se um fenômeno de cauda com uma métrica dominada pelo miolo da
distribuição.

Este notebook reformula o problema para medir diretamente o que foi encontrado.

## Alvo

> `y = 1` se `Δ₆ = taxa(t+6) − taxa(t) > limiar` — o segmento **deteriora significativamente**
> nos próximos seis meses.

Três consequências de desenho:

1. **Elimina o efeito de nível por construção.** Como o alvo é a *variação*, desaparece o
   componente transversal que inflava o R² global do estudo de regressão. Não há mais o problema
   do "R² *within* negativo": a métrica já é, por definição, intrassegmento.
2. **A persistência deixa de ser um baseline forte.** "Prever nenhuma mudança" não detecta evento
   algum. Os baselines passam a ser *momentum*, nível atual, crédito vencido e o ARIMA.
3. **Permite inferência com mais poder.** AUC admite o teste de **DeLong** para comparação de
   curvas, complementado aqui por um **bootstrap por bloco mensal** que respeita a dependência
   do painel.

## Decisões registradas *a priori* (antes de observar qualquer resultado)

| Decisão | Escolha | Justificativa |
|---|---|---|
| Limiar primário | **P80** das variações de 6 meses **no treino** | Data-driven mas sem vazamento; ≈20% de eventos |
| Sensibilidade | P70, P90 e +1 p.p. absoluto | Mostra que a conclusão não depende do corte |
| Variáveis | as do pipeline **+ `nivel_disp`** (último nível publicado) | Necessário para prever uma *variação* e torna a comparação com os baselines justa |
| Protocolo | split temporal + **embargo de horizonte + defasagem** (8 meses) | Idêntico ao notebook 04 |
| Métrica primária | **AUC-ROC** | Livre de limiar; inferência estabelecida |
| Métricas operacionais | AUC-PR, *precision@10%*, *lift* | O que um gestor de risco de fato usa |

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, brier_score_loss)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

BASE = Path().resolve().parent
PROC = BASE / 'data' / 'processed'
SEG_KEYS = ['cnae_secao', 'porte', 'modalidade']

with open(PROC / 'ml_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)
FEATURE_COLS = meta['feature_cols']
SPLIT_DATE   = pd.Timestamp(meta['split_date'])
HORIZONTE    = int(meta.get('horizonte', 6))

# --- Defasagem de publicacao (revisao A1) ---
# Gravada pelo notebook 03. Com lag > 0: as variaveis do SCR ja vem deslocadas; o nivel
# corrente CONHECIDO na origem e `nivel_disp` = taxa(t - lag); e o embargo passa a ser
# HORIZONTE + lag, porque o alvo de uma linha de treino precisa ja estar PUBLICADO na data
# da previsao (d + h + lag <= origem).
LAG_SCR = int(meta.get('lag_scr', 0))
NIVEL   = 'nivel_disp' if LAG_SCR else 'taxa_inadimplencia'
EMBARGO = HORIZONTE + LAG_SCR

df = (pd.read_parquet(PROC / 'ml_dataset.parquet')
        .sort_values('data_base').reset_index(drop=True))

# Variacao de 6 meses: alvo(t+6) menos o nivel REAL em t. O evento e definido sobre a serie
# verdadeira (e a quantidade economica de interesse); a defasagem afeta o que o modelo VE,
# nao o que ele tenta prever.
df['delta6'] = df['target'] - df['taxa_inadimplencia']

# --- Split temporal com embargo ---
embargo_lim = SPLIT_DATE - pd.DateOffset(months=EMBARGO)
mask_tr = df['data_base'] <  embargo_lim
mask_te = df['data_base'] >= SPLIT_DATE

# --- Limiar do evento: P80 das variacoes NO TREINO (sem vazamento) ---
Q_PRIMARIO = 0.80
LIMIAR = float(df.loc[mask_tr, 'delta6'].quantile(Q_PRIMARIO))
df['evento'] = (df['delta6'] > LIMIAR).astype(int)

# O nivel conhecido na origem entra como variavel (necessario p/ prever uma variacao)
FEATS = FEATURE_COLS + [NIVEL]

df_train, df_test = df[mask_tr].copy(), df[mask_te].copy()
X_train, y_train = df_train[FEATS], df_train['evento'].to_numpy()
X_test,  y_test  = df_test[FEATS],  df_test['evento'].to_numpy()

print(f'Defasagem de publicacao do SCR: {LAG_SCR} mes(es)  |  embargo: {EMBARGO} meses  |  '
      f'nivel conhecido: {NIVEL}')
print(f'Limiar (P{int(Q_PRIMARIO*100)} das variacoes de 6m no treino): '
      f'{LIMIAR:.4f}  ({LIMIAR*100:.2f} p.p.)')
print(f'Variaveis: {len(FEATS)} ({len(FEATURE_COLS)} do pipeline + nivel conhecido)')
print()
print(f'Treino: {len(df_train):>7,} linhas ({df_train.data_base.min().date()} -> '
      f'{df_train.data_base.max().date()})  eventos: {y_train.mean():6.2%}')
print(f'Teste:  {len(df_test):>7,} linhas ({df_test.data_base.min().date()} -> '
      f'{df_test.data_base.max().date()})  eventos: {y_test.mean():6.2%}')

## 1. Baselines

Na regressão, a persistência (`ŷ(t+6)=taxa(t)`) era um adversário forte por causa da
autocorrelação. Para prever *variação*, ela equivale a prever `Δ=0` — um escore constante, sem
poder de ordenação (AUC = 0,5). Os baselines relevantes passam a ser outros:

| Baseline | Escore | Hipótese |
|---|---|---|
| **Momentum** | `taxa(t) − taxa(t−3)` | Quem vem piorando continua piorando |
| **Nível atual** | `taxa(t)` | Nível alto sinaliza risco (ou reversão à média, se AUC < 0,5) |
| **Crédito vencido** | `vencido_lag1` | Indicador antecedente natural |
| **ARIMA(1,1,1)** | `previsão(t+6) − taxa(t)` | Baseline estatístico clássico, origem móvel |

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

def arima_rolling_forecast(df_all, split_date, seg_keys, horizonte=6, lag_pub=0,
                           col='taxa_inadimplencia'):
    """ARIMA(1,1,1) com origem movel (mesma funcao do notebook 04).

    Com `lag_pub` > 0 (defasagem de publicacao, revisao A1), na origem r o modelo so conhece a
    serie ate r - lag_pub — o ultimo mes PUBLICADO — e precisa prever horizonte + lag_pub passos
    para alcancar t+h. Com lag_pub = 0 a funcao e identica a versao original.
    """
    pred = pd.Series(np.nan, index=df_all.index, dtype=float)
    split = np.datetime64(pd.Timestamp(split_date))
    passos = horizonte + lag_pub
    for _, grp in df_all.groupby(seg_keys, observed=True):
        grp = grp.sort_values('data_base')
        y = grp[col].to_numpy(dtype=float)
        rows, dates, n = grp.index.to_numpy(), grp['data_base'].to_numpy(), len(grp)
        n_train = int((dates < split).sum())
        k = n_train - lag_pub                       # observacoes conhecidas na primeira origem
        if k < 12 or n_train >= n:
            continue
        try:
            ext = ARIMA(y[:k], order=(1, 1, 1)).fit()
        except Exception:
            continue
        for r in range(n_train - 1, n):             # r = origem da previsao
            try:
                while k < r - lag_pub + 1:          # incorpora o que foi publicado ate a origem
                    ext = ext.append([y[k]], refit=False)
                    k += 1
                if r + horizonte < n:
                    pred.loc[rows[r]] = ext.forecast(steps=passos)[passos - 1]
            except Exception:
                break
    return pred

print('Rodando ARIMA(1,1,1) rolling (alguns minutos)...')
pred_arima = arima_rolling_forecast(df, SPLIT_DATE, SEG_KEYS, horizonte=HORIZONTE,
                                    lag_pub=LAG_SCR)

# Escores dos baselines no conjunto de teste (maior = mais provavel deteriorar).
# Todos usam apenas informacao PUBLICADA na origem.
escores = {
    'Momentum (t vs t-3)': (df_test[NIVEL] - df_test['inadim_lag3']).to_numpy(),
    'Nivel atual':          df_test[NIVEL].to_numpy(),
    'Vencido t-1':          df_test['vencido_lag1'].to_numpy(),
    'ARIMA(1,1,1)':        (pred_arima.loc[df_test.index] - df_test[NIVEL]).to_numpy(),
}
if 'vol12_disp' in df_test.columns:        # referencial de volatilidade (revisao A2a)
    escores['Volatilidade 12m'] = df_test['vol12_disp'].to_numpy()
for nome_b, v in escores.items():
    n_ok = np.isfinite(v).sum()
    print(f'  {nome_b:22s} cobertura: {n_ok:,}/{len(v):,}')

## 2. Modelos de classificação

Mesma escada de complexidade do estudo de regressão, para manter a comparabilidade:
Regressão Logística, Árvore de Decisão, Random Forest, XGBoost e MLP.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Revisao A4a: variante da Regressao Logistica com one-hot nas categoricas + padronizacao.
CAT_COLS = [c for c in FEATS if c.endswith('_enc')]
NUM_COLS = [c for c in FEATS if c not in CAT_COLS]

def pre_onehot():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('scl', StandardScaler())]), NUM_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ])

def make_clfs():
    return {
        'Reg. Logistica': Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
            ('clf', LogisticRegression(max_iter=2000, random_state=42)),
        ]),
        'Reg. Logistica (one-hot)': Pipeline([
            ('pre', pre_onehot()),
            ('clf', LogisticRegression(max_iter=2000, random_state=42)),
        ]),
        'Arvore Decisao': Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('clf', DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=42)),
        ]),
        'Random Forest': Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('clf', RandomForestClassifier(n_estimators=300, max_depth=12,
                                           min_samples_leaf=10, n_jobs=-1, random_state=42)),
        ]),
        'XGBoost': Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('clf', XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=42, verbosity=0, eval_metric='logloss')),
        ]),
        'MLP': Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
            ('clf', MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=400,
                                  early_stopping=True, random_state=42)),
        ]),
    }

# Validacao cruzada temporal no treino (dados ja ordenados por data)
tscv = TimeSeriesSplit(n_splits=5)
print('Validacao cruzada temporal (AUC no treino):')
for nome, pipe in make_clfs().items():
    sc = cross_val_score(pipe, X_train, y_train, cv=tscv, scoring='roc_auc', n_jobs=-1)
    print(f'  {nome:26s} AUC CV: {sc.mean():.4f} +/- {sc.std():.4f}')

In [ ]:
# Treino e predicao no teste
probs = {}
for nome, pipe in make_clfs().items():
    pipe.fit(X_train, y_train)
    probs[nome] = pipe.predict_proba(X_test)[:, 1]
    print(f'{nome:16s} treinado')

# Junta modelos + baselines num unico dicionario de escores
todos = {**probs, **escores}
print(f'\nTotal avaliado: {len(todos)} preditores '
      f'({len(probs)} modelos + {len(escores)} baselines)')

## 3. Avaliação

- **AUC-ROC** — métrica primária, livre de limiar.
- **AUC-PR (average precision)** — mais informativa que a ROC quando a classe positiva é minoritária.
- **Precision@10%** — dos 10% de segmentos marcados como mais arriscados, quantos de fato
  deterioraram. É a métrica operacional de um sistema de alerta.
- **Lift@10%** — quantas vezes melhor que marcar 10% ao acaso.
- **Brier** — calibração (apenas para modelos que produzem probabilidade).

In [ ]:
def avaliar(y, s, k=0.10, prob=False):
    ok = np.isfinite(s)
    y_, s_ = y[ok], s[ok]
    n_top = max(1, int(len(s_) * k))
    top = np.argsort(-s_)[:n_top]
    prec_k = y_[top].mean()
    base = y_.mean()
    return {
        'AUC-ROC':   roc_auc_score(y_, s_),
        'AUC-PR':    average_precision_score(y_, s_),
        'P@10%':     prec_k,
        'Recall@10%': y_[top].sum() / y_.sum(),
        'Lift@10%':  prec_k / base,
        'Brier':     brier_score_loss(y_, s_) if prob else np.nan,
        'n':         int(ok.sum()),
    }

linhas = {}
for nome, s in todos.items():
    linhas[nome] = avaliar(y_test, np.asarray(s, dtype=float), prob=(nome in probs))

res = (pd.DataFrame(linhas).T
         .sort_values('AUC-ROC', ascending=False)
         .round({'AUC-ROC': 4, 'AUC-PR': 4, 'P@10%': 4,
                 'Recall@10%': 4, 'Lift@10%': 2, 'Brier': 4}))
res.index.name = 'Preditor'
print(f'Taxa-base de eventos no teste: {y_test.mean():.2%}')
print()
print(res.to_string())
res.to_csv(PROC / 'resultados_classificacao.csv')
print()
print('Salvo: resultados_classificacao.csv')

In [ ]:
# === Revisao A2b: limiar RELATIVO por segmento ===
# Um limiar absoluto unico (P80 de delta6) marca como "evento" sobretudo os segmentos que sao
# naturalmente volateis — e esses podem ser identificaveis so pela volatilidade historica.
# Aqui o evento e: delta6 > K_REL * sigma_seg, onde sigma_seg e o desvio-padrao das variacoes
# de 6 meses DO PROPRIO SEGMENTO, estimado apenas no treino. Pergunta: a deterioracao foi grande
# PARA AQUELE segmento? Quanto do ganho do ML sobre o melhor referencial sobrevive?
K_REL = 1.0
gid_all   = df[SEG_KEYS].astype(str).agg('|'.join, axis=1)
sigma_seg = df.loc[mask_tr, 'delta6'].groupby(gid_all[mask_tr]).std()
sig = gid_all.map(sigma_seg).fillna(float(df.loc[mask_tr, 'delta6'].std()))
df['evento_rel'] = (df['delta6'] > K_REL * sig).astype(int)
y_train_rel = df.loc[mask_tr, 'evento_rel'].to_numpy()
y_test_rel  = df.loc[mask_te, 'evento_rel'].to_numpy()
print(f'Limiar relativo: delta6 > {K_REL:g} x sigma_seg (sigma do segmento no treino; '
      f'mediana {sigma_seg.median()*100:.2f} p.p., {int(sigma_seg.isna().sum())} segmentos sem historico)')
print(f'Eventos: treino {y_train_rel.mean():.2%} | teste {y_test_rel.mean():.2%}')
print()

probs_rel = {}
for nome, pipe in make_clfs().items():
    if nome not in ('Random Forest', 'XGBoost', 'Reg. Logistica', 'Reg. Logistica (one-hot)'):
        continue
    pipe.fit(X_train, y_train_rel)
    probs_rel[nome] = pipe.predict_proba(X_test)[:, 1]
    print(f'{nome:26s} treinado (alvo relativo)')

todos_rel = {**probs_rel, **escores}
linhas_rel = {nome: avaliar(y_test_rel, np.asarray(s, dtype=float), prob=(nome in probs_rel))
              for nome, s in todos_rel.items()}
res_rel = (pd.DataFrame(linhas_rel).T
             .sort_values('AUC-ROC', ascending=False)
             .round({'AUC-ROC': 4, 'AUC-PR': 4, 'P@10%': 4,
                     'Recall@10%': 4, 'Lift@10%': 2, 'Brier': 4}))
res_rel.index.name = 'Preditor'
print()
print(f'=== Alvo RELATIVO (k={K_REL:g}) — taxa-base no teste: {y_test_rel.mean():.2%} ===')
print(res_rel.to_string())
res_rel.to_csv(PROC / 'resultados_classificacao_relativo.csv')
print()
print('Salvo: resultados_classificacao_relativo.csv')

In [ ]:
# === Revisao A2c: AUC calculada DENTRO de cada mes e depois promediada ===
# A AUC agrupada pode ser inflada por ordenacao ENTRE meses (um mes com muitos eventos e escores
# altos para todos). A AUC mensal mede apenas a capacidade de ordenar os segmentos dentro de um
# mesmo mes — o que um gestor faz na pratica. Reporta-se para o alvo P80 e para o alvo relativo,
# com IC bootstrap (reamostragem de meses, diferencas pareadas por mes) da vantagem sobre o
# referencial de credito vencido.
def aucs_por_mes(y, s, meses, min_n=20):
    s = np.asarray(s, dtype=float)
    out = {}
    for m in np.unique(meses):
        ii = meses == m
        yy, ss = y[ii], s[ii]
        ok = np.isfinite(ss)
        if ok.sum() < min_n or yy[ok].min() == yy[ok].max():
            continue
        out[m] = roc_auc_score(yy[ok], ss[ok])
    return out

def ic_dif_mensal(a_mod, a_base, B=600, seed=42):
    """IC 95% bootstrap (meses com reposicao) da media das diferencas mensais de AUC."""
    comuns = sorted(set(a_mod) & set(a_base))
    d = np.array([a_mod[m] - a_base[m] for m in comuns])
    rng = np.random.default_rng(seed)
    bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(B)])
    return float(d.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

meses_te = df_test['data_base'].to_numpy()
lin_m = []
for rot_alvo, dic, yy, tab in [('P80 absoluto', todos, y_test, res),
                               (f'relativo {K_REL:g}*sigma_seg', todos_rel, y_test_rel, res_rel)]:
    por_mes = {nome: aucs_por_mes(yy, s, meses_te) for nome, s in dic.items()}
    base = por_mes['Vencido t-1']
    for nome, am in por_mes.items():
        vals = np.array(list(am.values()))
        linha = {'Alvo': rot_alvo, 'Preditor': nome,
                 'AUC agrupada': round(float(tab.loc[nome, 'AUC-ROC']), 4),
                 'AUC media mensal': round(float(vals.mean()), 4),
                 'dp mensal': round(float(vals.std(ddof=1)), 4), 'meses': int(len(vals))}
        if nome != 'Vencido t-1':
            dm_, lo, hi = ic_dif_mensal(am, base)
            linha['dif vs vencido'] = round(dm_, 4)
            linha['IC95 dif vs vencido'] = f'[{lo:+.4f}, {hi:+.4f}]'
        lin_m.append(linha)

df_aucm = pd.DataFrame(lin_m).sort_values(['Alvo', 'AUC media mensal'], ascending=[True, False])
print('=== AUC agrupada vs AUC media por mes (teste 2022-2024); IC bootstrap de 600 reamostragens de meses ===')
print(df_aucm.to_string(index=False))
df_aucm.to_csv(PROC / 'resultados_classificacao_auc_mensal.csv', index=False)
print()
print('Salvo: resultados_classificacao_auc_mensal.csv')

## 4. Inferência sobre as AUCs

Duas abordagens complementares:

- **DeLong (1988)** — teste padrão para comparar duas AUCs correlacionadas. Assume observações
  independentes, o que **não** vale num painel; tende portanto a *subestimar* a variância.
- **Bootstrap por bloco mensal** — reamostra meses inteiros com reposição, preservando a
  dependência transversal dentro de cada mês. É a inferência honesta para estes dados.

Reporta-se ambas: quando concordam, a conclusão é robusta; quando divergem, prevalece o bootstrap.

In [ ]:
from scipy import stats as sps

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2

def delong(y, s1, s2):
    """Teste de DeLong para duas AUCs correlacionadas (versao rapida, Sun & Xu 2014)."""
    ok = np.isfinite(s1) & np.isfinite(s2)
    y_, S = y[ok], np.vstack([s1[ok], s2[ok]])
    pos, neg = S[:, y_ == 1], S[:, y_ == 0]
    m, n = pos.shape[1], neg.shape[1]
    tx = np.array([_midrank(pos[r]) for r in range(2)])
    ty = np.array([_midrank(neg[r]) for r in range(2)])
    tz = np.array([_midrank(np.concatenate([pos[r], neg[r]])) for r in range(2)])
    aucs = (tz[:, :m].sum(axis=1) - m * (m + 1) / 2) / (m * n)
    v_pos = (tz[:, :m] - tx) / n
    v_neg = 1 - (tz[:, m:] - ty) / m
    Sig = np.cov(v_pos) / m + np.cov(v_neg) / n
    var = Sig[0, 0] + Sig[1, 1] - 2 * Sig[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(max(var, 1e-16))
    return aucs[0], aucs[1], z, 2 * (1 - sps.norm.cdf(abs(z)))

def boot_mensal(y, s1, s2, meses, B=600, seed=42):
    """IC bootstrap da diferenca de AUC reamostrando MESES inteiros."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(meses)
    idx_por_mes = {mm: np.where(meses == mm)[0] for mm in uniq}
    difs = []
    for _ in range(B):
        sel = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_por_mes[mm] for mm in sel])
        yy = y[idx]
        if yy.min() == yy.max():
            continue
        a, b = s1[idx], s2[idx]
        ok = np.isfinite(a) & np.isfinite(b)
        if ok.sum() < 50 or yy[ok].min() == yy[ok].max():
            continue
        difs.append(roc_auc_score(yy[ok], a[ok]) - roc_auc_score(yy[ok], b[ok]))
    difs = np.array(difs)
    return difs.mean(), np.percentile(difs, 2.5), np.percentile(difs, 97.5), (difs <= 0).mean()

meses_te = df_test['data_base'].to_numpy()
melhor = res.index[0]
s_melhor = np.asarray(todos[melhor], dtype=float)

comparacoes = [c for c in res.index[1:] if c != melhor][:5]
out = []
for c in comparacoes:
    s_c = np.asarray(todos[c], dtype=float)
    a1, a2, z, p = delong(y_test, s_melhor, s_c)
    md, lo, hi, pboot = boot_mensal(y_test, s_melhor, s_c, meses_te)
    out.append({'Comparacao': f'{melhor} vs {c}',
                f'AUC {melhor}': round(a1, 4), 'AUC outro': round(a2, 4),
                'dif': round(a1 - a2, 4),
                'p (DeLong)': round(p, 4),
                'IC95 boot': f'[{lo:+.3f}, {hi:+.3f}]',
                'p (boot)': round(pboot, 4)})

df_inf = pd.DataFrame(out)
print(f'Melhor preditor por AUC: {melhor}')
print()
print(df_inf.to_string(index=False))
df_inf.to_csv(PROC / 'resultados_classificacao_inferencia.csv', index=False)
print()
print('Salvo: resultados_classificacao_inferencia.csv')

## 5. Sensibilidade ao limiar

A conclusão não pode depender do corte escolhido. Reavalia-se com P70, P80, P90 e um limiar
absoluto de +1 ponto percentual (todos com o limiar estimado **no treino**).

In [ ]:
cenarios = {'P70': ('q', 0.70), 'P80': ('q', 0.80),
            'P90': ('q', 0.90), '+1 p.p.': ('abs', 0.01)}

sens = []
for nome_c, (tipo, val) in cenarios.items():
    lim = float(df.loc[mask_tr, 'delta6'].quantile(val)) if tipo == 'q' else val
    ytr = (df.loc[mask_tr, 'delta6'] > lim).astype(int).to_numpy()
    yte = (df.loc[mask_te, 'delta6'] > lim).astype(int).to_numpy()
    linha = {'Cenario': nome_c, 'limiar (p.p.)': round(lim * 100, 2),
             'eventos treino': f'{ytr.mean():.1%}', 'eventos teste': f'{yte.mean():.1%}'}
    for nome, pipe in make_clfs().items():
        if nome not in ('Random Forest', 'XGBoost', 'Reg. Logistica'):
            continue
        pipe.fit(X_train, ytr)
        linha[nome] = round(roc_auc_score(yte, pipe.predict_proba(X_test)[:, 1]), 4)
    # melhor baseline por AUC no cenario
    mb, ab = None, 0.5
    for k, s in escores.items():
        s = np.asarray(s, dtype=float); ok = np.isfinite(s)
        a = roc_auc_score(yte[ok], s[ok])
        a = max(a, 1 - a)   # baseline pode ser preditor invertido
        if a > ab:
            mb, ab = k, a
    linha['melhor baseline'] = f'{mb} ({ab:.4f})'
    sens.append(linha)

df_sens = pd.DataFrame(sens)
print('=== Sensibilidade ao limiar (AUC-ROC no teste) ===')
print(df_sens.to_string(index=False))
df_sens.to_csv(PROC / 'resultados_classificacao_sensibilidade.csv', index=False)
print()
print('Salvo: resultados_classificacao_sensibilidade.csv')

## 6. Curvas ROC e Precisão-Revocação

In [ ]:
# Paleta categorica validada (ordem fixa dos slots; separacao CVD dE 24.2)
COR = {'Random Forest': '#2a78d6', 'XGBoost': '#1baf7a', 'Reg. Logistica': '#eda100',
       'Reg. Logistica (one-hot)': '#b07a00',
       'Arvore Decisao': '#4a3aa7', 'MLP': '#e34948',
       'ARIMA(1,1,1)': '#008300', 'Momentum (t vs t-3)': '#eb6834',
       'Nivel atual': '#898781', 'Vencido t-1': '#e87ba4', 'Volatilidade 12m': '#7a5c3a'}
# Rotulos coerentes com o texto do artigo: os referenciais usam o ultimo valor PUBLICADO
ROTULO = {'Vencido t-1': 'Vencido (ultimo publ.)', 'Nivel atual': 'Nivel (ultimo publ.)',
          'Momentum (t vs t-3)': 'Momentum (3 meses)', 'Volatilidade 12m': 'Volatilidade 12m',
          'Reg. Logistica': 'Reg. Logistica (rotulo)', 'Arvore Decisao': 'Arvore de Decisao'}
rot = lambda n: ROTULO.get(n, n)
SURF, GRID, AXIS = '#fcfcfb', '#e1e0d9', '#c3c2b7'
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'

# Revisao C4: as curvas dos referenciais entram na figura (vencido, nivel e volatilidade),
# ja que sao os adversarios efetivamente comparados no texto.
mostrar = list(probs.keys()) + ['ARIMA(1,1,1)', 'Momentum (t vs t-3)',
                                'Vencido t-1', 'Nivel atual']
if 'Volatilidade 12m' in todos:
    mostrar.append('Volatilidade 12m')
TRACEJADO = ('Vencido t-1', 'Nivel atual', 'Momentum (t vs t-3)', 'Volatilidade 12m')
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=SURF)

ax = axes[0]; ax.set_facecolor(SURF)
for nome in mostrar:
    s = np.asarray(todos[nome], dtype=float); ok = np.isfinite(s)
    fpr, tpr, _ = roc_curve(y_test[ok], s[ok])
    ls = '--' if nome in TRACEJADO else '-'
    ax.plot(fpr, tpr, ls, color=COR.get(nome, '#555555'), linewidth=2,
            label=f'{rot(nome)} ({roc_auc_score(y_test[ok], s[ok]):.3f})')
ax.plot([0, 1], [0, 1], ':', color=AXIS, linewidth=1, label='acaso (0.500)')
ax.set_xlabel('Falso positivo', fontsize=10, color=INK2)
ax.set_ylabel('Verdadeiro positivo', fontsize=10, color=INK2)
ax.set_title('Curva ROC — deterioração em t+6', fontsize=12, color=INK, pad=10)
ax.legend(fontsize=7, frameon=False, loc='lower right', labelcolor=INK2)
ax.grid(True, color=GRID, linewidth=0.7)

ax = axes[1]; ax.set_facecolor(SURF)
base = y_test.mean()
for nome in mostrar:
    s = np.asarray(todos[nome], dtype=float); ok = np.isfinite(s)
    pr, rc, _ = precision_recall_curve(y_test[ok], s[ok])
    ls = '--' if nome in TRACEJADO else '-'
    ax.plot(rc, pr, ls, color=COR.get(nome, '#555555'), linewidth=2,
            label=f'{rot(nome)} ({average_precision_score(y_test[ok], s[ok]):.3f})')
ax.axhline(base, linestyle=':', color=AXIS, linewidth=1, label=f'taxa-base ({base:.3f})')
ax.set_xlabel('Revocação', fontsize=10, color=INK2)
ax.set_ylabel('Precisão', fontsize=10, color=INK2)
ax.set_title('Curva Precisão-Revocação', fontsize=12, color=INK, pad=10)
ax.legend(fontsize=7, frameon=False, loc='upper right', labelcolor=INK2)
ax.grid(True, color=GRID, linewidth=0.7)

for a in axes:
    for s_ in ['top', 'right']:
        a.spines[s_].set_visible(False)
    for s_ in ['left', 'bottom']:
        a.spines[s_].set_color(AXIS)
    a.tick_params(colors=MUTED, labelsize=9)

plt.tight_layout()
plt.savefig(PROC / 'resultados_classificacao_curvas.png', dpi=150,
            bbox_inches='tight', facecolor=SURF)
plt.show()
print('Salvo: resultados_classificacao_curvas.png')